In [13]:
import pandas as pd
import numpy as np
from statsbombpy import sb

# 1. Load Data
match_id = 9774
# fetching as 'dict' gives us the raw structure we need to extract nested objects cleanly
events_dict = sb.events(match_id=match_id, fmt="dict")
events_list = list(events_dict.values())

# 2. Pre-processing Helpers

# A. Create a lookup dictionary for Related Events (ID -> Type Name)
# We need this to populate the 'related_events' column with types, not just IDs.
id_to_type_lookup = {e['id']: e['type']['name'] for e in events_list}

def enrich_related_events(related_ids):
    """LOOKUP function: Takes a list of UUIDs and returns a list of dictionaries with ID and Type."""
    if not isinstance(related_ids, list):
        return np.nan
    
    enriched = []
    for uuid in related_ids:
        # Get type name if exists, else 'Unknown'
        type_name = id_to_type_lookup.get(uuid, "Unknown")
        enriched.append({'id': uuid, 'type': type_name})
    return enriched

# B. Logic to extract the specific event attributes based on the event type
def get_event_attributes(row):
    """
    Extracts the nested object corresponding to the event type.
    E.g., if type is 'Pass', extract the dictionary under the key 'pass'.
    See 'Event Type Objects' in docs.
    """
    evt_type_name = row['type']['name']
    
    # Convert "Ball Receipt*" to "ball_receipt", "Goal Keeper" to "goalkeeper", etc.
    # The keys in the JSON are usually snake_case versions of the type name.
    key_slug = evt_type_name.lower().replace(' ', '_').replace('*', '')
    
    # Handle edge case where "Goal Keeper" might be "goalkeeper" in keys
    if key_slug == 'goal_keeper': key_slug = 'goalkeeper'
    
    return row.get(key_slug, np.nan)

def determine_half(location):
    """
    Determines if the event is in 'Own Half' or 'Opponent Half'.
    Center is (60,40).
    """
    if not isinstance(location, list) or len(location) < 2:
        return np.nan
    
    x = location[0]
    if x <= 60:
        return "Own Half"
    else:
        return "Opponent Half"

# 3. Construct the DataFrame
# We loop through the list once to build the structured data
data_rows = []

for event in events_list:
    # Basic Extraction
    row_data = {
        'event_id': event.get('id'),
        'event_index': event.get('index'),
        'event_period': event.get('period'),
        'event_timestamp': event.get('timestamp'),
        'event_minute': event.get('minute'),
        'event_second': event.get('second'),
        'event_type': event.get('type', {}).get('name'),
        # Flattened JSON of event specific attributes 
        'event_type_attributes': get_event_attributes(event),
        'team': event.get('team', {}).get('name'),
        'player': event.get('player', {}).get('name'),
        'location': event.get('location', np.nan),
        'under_pressure': event.get('under_pressure', False), # 
        'out': event.get('out', False),                       # 
        'related_events_raw': event.get('related_events')     # 
    }
    data_rows.append(row_data)

df = pd.DataFrame(data_rows)

# 4. Apply Derived Columns

# Combine minute and second
df['event_match_time'] = df.apply(lambda x: f"{x['event_minute']}:{x['event_second']:02d}", axis=1)

# Enrich Related Events (Search for event info)
df['related_events'] = df['related_events_raw'].apply(enrich_related_events)

# Calculate Half from Location
df['half'] = df['location'].apply(determine_half)

# 5. Final Selection & Ordering
final_columns = [
    # 'event_id',
    'event_index',
    # 'event_period',
    # 'event_timestamp',
    'event_match_time',
    'event_type',
    'event_type_attributes', # The flattened JSON/Dict you requested
    'team',
    'player',
    'location',
    # 'half',
    'under_pressure',
    'out',
    'related_events'
]

df_final = df[final_columns].sort_values('event_index')


df_final.head(10)

/opt/homebrew/Caskroom/miniconda/base/envs/mlp/lib/python3.12/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,event_index,event_match_time,event_type,event_type_attributes,team,player,location,under_pressure,out,related_events
0,1,0:00,Starting XI,NaN,Barcelona,NaN,NaN,False,False,NaN
1,2,0:00,Starting XI,NaN,Deportivo Alavés,NaN,NaN,False,False,NaN
2,3,0:00,Half Start,NaN,Deportivo Alavés,NaN,NaN,False,False,[{'id': 'ab533034-8ad3-440f-8fdd-5ac2ec54a04f'...
3,4,0:00,Half Start,NaN,Barcelona,NaN,NaN,False,False,[{'id': '5c327f3b-19cf-4325-991e-eb67f6822302'...
4,5,0:01,Pass,"{'recipient': {'id': 6736, 'name': 'Álvaro Med...",Deportivo Alavés,John Guidetti,"[61.0, 40.1]",False,False,[{'id': '7c251896-5c01-4412-a4a4-5b72e9f55c97'...
5,6,0:01,Ball Receipt*,NaN,Deportivo Alavés,Álvaro Medrán Just,"[43.9, 42.1]",False,False,[{'id': '5d7b9078-4141-476f-acde-9a9a631edc1c'...
6,7,0:01,Carry,"{'end_location': [55.7, 41.7]}",Deportivo Alavés,Álvaro Medrán Just,"[43.9, 42.1]",False,False,[{'id': '7c251896-5c01-4412-a4a4-5b72e9f55c97'...
7,8,0:03,Pass,"{'recipient': {'id': 6038, 'name': 'John Guide...",Deportivo Alavés,Álvaro Medrán Just,"[55.7, 41.7]",False,False,[{'id': '8e56db83-93cf-4713-ad8e-fa6e46173183'...
8,9,0:06,Ball Receipt*,"{'outcome': {'id': 9, 'name': 'Incomplete'}}",Deportivo Alavés,John Guidetti,"[93.3, 23.7]",False,False,[{'id': 'baa798a9-6108-465e-8503-54e4e0b33474'...
9,10,0:06,Duel,"{'type': {'id': 10, 'name': 'Aerial Lost'}}",Deportivo Alavés,John Guidetti,"[93.3, 23.7]",True,False,[{'id': '8e56db83-93cf-4713-ad8e-fa6e46173183'...


In [15]:
# 1. Explode the list of related events into individual rows
df_exploded = df_final.explode('related_events').reset_index(drop=True)

# 2. Normalize only the exploded dictionaries
# We handle non-dict values (like NaN) to prevent errors
related_details = pd.json_normalize(
    df_exploded['related_events'].apply(lambda x: x if isinstance(x, dict) else {})
)

# 3. Rename columns to avoid overlap
related_details.columns = [f"related_{col}" for col in related_details.columns]

# 4. Concatenate safely now that indices are aligned and unique
df_related_expanded = pd.concat([df_exploded, related_details], axis=1)

# Display results
print(df_related_expanded[['event_type', 'related_id', 'related_type']].head(10))

      event_type                            related_id   related_type
0    Starting XI                                   NaN            NaN
1    Starting XI                                   NaN            NaN
2     Half Start  ab533034-8ad3-440f-8fdd-5ac2ec54a04f     Half Start
3     Half Start  5c327f3b-19cf-4325-991e-eb67f6822302     Half Start
4           Pass  7c251896-5c01-4412-a4a4-5b72e9f55c97  Ball Receipt*
5  Ball Receipt*  5d7b9078-4141-476f-acde-9a9a631edc1c           Pass
6          Carry  7c251896-5c01-4412-a4a4-5b72e9f55c97  Ball Receipt*
7          Carry  baa798a9-6108-465e-8503-54e4e0b33474           Pass
8           Pass  8e56db83-93cf-4713-ad8e-fa6e46173183           Pass
9           Pass  d6480e42-2fed-409a-8d1c-065ab4e171cf  Ball Receipt*
